# EMD / CMIP-LD Development Notebook

Testing the ReportBuilder pipeline — no external CV config needed.
The builder infers CV field graphs from the expanded folder items themselves.

In [18]:
import sys
# sys.path.insert(0, '/Users/daniel.ellis/WIPwork/CMIP-LD')
# sys.path.insert(0, '/Users/daniel.ellis/WIPwork/Essential-Model-Documentation')

import importlib, json, pprint
pp = pprint.PrettyPrinter(indent=2).pprint

TEST_ITEM = {
    "validation_key": "tempgrid_matthew-mizielinski-1776857536",
    "ui_label": "Horizontal grid cell with a regular latitude longitude grid type and 1.875 x 1.25 degree resolution.",
    "description": "",
    "alias": [],
    "grid_mapping": "latitude-longitude",
    "grid_type": "regular-latitude-longitude",
    "n_cells": 27648,
    "region": [
        "global"
    ],
    "southernmost_latitude": -89.375,
    "temporal_refinement": "static",
    "truncation_method": "",
    "truncation_number": "",
    "units": "degree",
    "westernmost_longitude": 0,
    "x_resolution": 1.875,
    "y_resolution": 1.25,
    "@context": "_context",
    "@type": [
        "emd",
        "wcrp:horizontal_grid_cell",
        "esgvoc:HorizontalGridCell"
    ],
    "@id": "tempgrid-matthew-mizielinski-1776857536"
}
#     'validation_key': 'tempgrid_wolfiex-1777283038',
#     'ui_label': 'Horizontal grid cell with a hierarchical discrete global grid grid type and 1 x 1 km resolution.',
#     'description': 'TEST TEST TEST',
#     'alias': ['item2', ' item3'],
#     'grid_mapping': 'lambert-azimuthal-equal-area',
#     'grid_type': 'hierarchical-discrete-global-grid',
#     'n_cells': 1,
#     'region': ['southern-hemisphere'],
#     'southernmost_latitude': 1,
#     'temporal_refinement': 'dynamically-stretched',
#     'truncation_method': 'triangular',
#     'truncation_number': 1,
#     'units': 'km',
#     'westernmost_longitude': 1,
#     'x_resolution': 1,
#     'y_resolution': 1,
#     '@context': '_context',
#     '@type': ['emd', 'wcrp:horizontal_grid_cell', 'esgvoc:HorizontalGridCell'],
#     '@id': 'tempgrid-wolfiex-1777283038'
# }
print('Test item loaded.')

Test item loaded.


## 1. Load Folder Graph & Inspect Expanded Items
At depth=4, folder items have CV field values expanded to full `@id` URIs.
This is where the CV graph URLs are inferred from.

In [19]:
import cmipld
import cmipld.utils.similarity.report_builder as rb_mod
importlib.reload(rb_mod)

from cmipld.utils.similarity.graph_loader import GraphLoader

folder_url = 'emd:horizontal_grid_cell'
print(f'Loading {folder_url}/_graph.json at depth=4 ...')
with cmipld.ensure_remote():
    data = cmipld.expand(f'{folder_url}/_graph.json', depth=4)

loader = GraphLoader(folder_url, graph_data=data)
folder_items = loader.items
print(f'Loaded {len(folder_items)} folder items\n')

# Show linked fields in the first item
print('Linked fields in first item (these tell us which fields are CV fields):')
for k, v in folder_items[0].items():
    if isinstance(v, dict) and '@id' in v:
        print(f'  {k}: {v["@id"]}')

Loading emd:horizontal_grid_cell/_graph.json at depth=4 ...
[Cache MISS] emd:horizontal_grid_cell/_graph.json (depth=4)
Loaded 19 folder items

Linked fields in first item (these tell us which fields are CV fields):
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/grid_mapping: https://constants.mipcvs.dev/grid_mapping/latitude-longitude
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/grid_type: https://constants.mipcvs.dev/grid_type/regular-latitude-longitude
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/region: https://constants.mipcvs.dev/region/global
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/temporal_refinement: https://constants.mipcvs.dev/temporal_refinement/static
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/truncation_method: https://constants.mipcvs.dev/truncation_method/
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/units: https://constants.mipcvs.dev/units/degree


## 2. Infer CV Graph URLs from Folder Items
`_infer_cv_graphs_from_folder` scans the existing items and derives the graph URL
from the directory part of each `@id` URI.

In [20]:
from cmipld.utils.similarity.report_builder import _infer_cv_graphs_from_folder

field_graphs = _infer_cv_graphs_from_folder(folder_items)
print(f'Inferred {len(field_graphs)} CV field graph(s):')
for field, url in field_graphs.items():
    print(f'  {field}: {url}')

Inferred 6 CV field graph(s):
  grid_mapping: https://constants.mipcvs.dev/grid_mapping/_graph.json
  grid_type: https://constants.mipcvs.dev/grid_type/_graph.json
  region: https://constants.mipcvs.dev/region/_graph.json
  temporal_refinement: https://constants.mipcvs.dev/temporal_refinement/_graph.json
  units: https://constants.mipcvs.dev/units/_graph.json
  truncation_method: https://constants.mipcvs.dev/truncation_method/_graph.json


## 3. Fetch Each CV Graph & Check Valid Values
`_fetch_cv_graph` retrieves the full list of valid values for a field.

In [21]:
from cmipld.utils.similarity.report_builder import _fetch_cv_graph

for field, graph_url in field_graphs.items():
    lookup = _fetch_cv_graph(graph_url)
    submitted = TEST_ITEM.get(field, '')
    if isinstance(submitted, list): submitted = submitted[0] if submitted else ''
    # Try probes
    probes = [submitted, submitted.lower(), submitted.replace('-','_'), submitted.replace('_','-')]
    resolved_uri = next((lookup[p] for p in probes if p in lookup), None)
    status = '✅' if resolved_uri else '❌'
    print(f'{status} {field}: {submitted!r}')
    if resolved_uri:
        print(f'    → {resolved_uri}')
    else:
        available = sorted({v.split("/")[-1] for v in lookup.values()})[:5]
        print(f'    Valid values (sample): {available}')

[Cache MISS] https://constants.mipcvs.dev/grid_mapping/_graph.json (depth=2)
✅ grid_mapping: 'latitude-longitude'
    → https://constants.mipcvs.dev/grid_mapping/latitude-longitude
[Cache MISS] https://constants.mipcvs.dev/grid_type/_graph.json (depth=2)
✅ grid_type: 'regular-latitude-longitude'
    → https://constants.mipcvs.dev/grid_type/regular-latitude-longitude
[Cache MISS] https://constants.mipcvs.dev/region/_graph.json (depth=2)
✅ region: 'global'
    → https://constants.mipcvs.dev/region/global
[Cache MISS] https://constants.mipcvs.dev/temporal_refinement/_graph.json (depth=2)
✅ temporal_refinement: 'static'
    → https://constants.mipcvs.dev/temporal_refinement/static
[Cache MISS] https://constants.mipcvs.dev/units/_graph.json (depth=2)
✅ units: 'degree'
    → https://constants.mipcvs.dev/units/degree
[Cache MISS] https://constants.mipcvs.dev/truncation_method/_graph.json (depth=2)
❌ truncation_method: ''
    Valid values (sample): ['rhomboidal', 'triangular']


## 4. Test `_build_links_from_folder` — Self-Contained
Returns `(field_links, field_graphs)` — no external config needed.

In [22]:
from cmipld.utils.similarity.report_builder import _build_links_from_folder

field_links, field_graphs = _build_links_from_folder(TEST_ITEM, folder_items)

print(f'{len(field_links)}/{len(field_graphs)} links resolved:\n')

for field, url in field_graphs.items():
    val = TEST_ITEM.get(field, '')
    if isinstance(val, list): val = val[0] if val else ''
    
    if field in field_links:
        print(f'  ✅ {field}: {val!r} → {field_links[field].split("/")[-1]}')
    else:
        print(f'  ❌ {field}: {val!r} — not found in graph')

[Cache HIT] https://constants.mipcvs.dev/grid_mapping/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/grid_type/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/region/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/temporal_refinement/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/units/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/truncation_method/_graph.json (depth=2)
5/6 links resolved:

  ✅ grid_mapping: 'latitude-longitude' → latitude-longitude
  ✅ grid_type: 'regular-latitude-longitude' → regular-latitude-longitude
  ✅ region: 'global' → global
  ✅ temporal_refinement: 'static' → static
  ✅ units: 'degree' → degree
  ❌ truncation_method: '' — not found in graph


In [23]:
folder_items

[{'@id': 'https://emd.mipcvs.dev/horizontal_grid_cell/g115',
  '@type': ['wcrp:horizontal_grid_cell',
   'esgvoc:HorizontalGridCell',
   'https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/emd'],
  'https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/description': {'@value': ''},
  'https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/grid_mapping': {'@id': 'https://constants.mipcvs.dev/grid_mapping/latitude-longitude',
   '@type': ['https://constants.mipcvs.dev/docs/contents/GridMapping/universal',
    'wcrp:grid_mapping',
    'esgvoc:GridMapping',
    'https://constants.mipcvs.dev/docs/contents/GridMapping/constants'],
   'https://constants.mipcvs.dev/docs/contents/GridMapping/description': {'@value': 'A projection that treats latitude and longitude as planar X and Y coordinates, as defined by the CF conventions.'},
   'https://constants.mipcvs.dev/docs/contents/GridMapping/ui_label': {'@value': 'Latitude-Longitude'},
   'https://constants.mipcvs.dev/docs/contents/GridMap

## 5. Fraction Logic
Replicate what `_link_section` will display.

In [24]:
from cmipld.utils.similarity.report_builder import _is_report_skip

total_cv = sum(
    1 for k, v in TEST_ITEM.items()
    if k in field_graphs
    and not _is_report_skip(k)
    and v not in ('', None, [], {})
)
resolved = len(field_links)
fraction = f'{resolved}/{total_cv}'
pct = f'{resolved / total_cv * 100:.0f}%' if total_cv else '—'

print(f'Display: CV links resolved: {fraction} ({pct})')

Display: CV links resolved: 5/5 (100%)


## 6. Full ReportBuilder Run — No Config Needed

In [ ]:
importlib.reload(rb_mod)
from cmipld.utils.similarity.report_builder import ReportBuilder

report = ReportBuilder(
    folder_url='emd:horizontal_grid_cell',
    kind='horizontal_grid_cell',
    item=TEST_ITEM,
    link_threshold=75.0,
).build()

print(f'Report generated: {len(report)} chars')

In [ ]:
from IPython.display import Markdown
Markdown(report)

## Submission Review — `tempgrid-matthew-mizielinski-1776857536`

> [!IMPORTANT]  
> This report is for use of reviewers only! 
> It is not intended to be used by submitters and may contain technical details and internal links that are not meaningful outside the review process. 

| Property | Value |
|----------|-------|
| **Type** | `horizontal_grid_cell` |
| **Submitted ID** | `tempgrid-matthew-mizielinski-1776857536` |
| **Generated** | 2026-04-28 00:35 UTC |


---

### 1. Field Status

- [x] `southernmost_latitude`: `-89.375`
- [x] `truncation_method`: _not submitted_
- [x] `truncation_number`: _not submitted_
- [x] `westernmost_longitude`: `0`
- [ ] `description`: _not submitted_
- [ ] `grid_mapping`: `latitude-longitude`
- [ ] `grid_type` _(required)_: `regular-latitude-longitude`
- [ ] `n_cells`: `27648`
- [ ] `region` _(required)_: `global`
- [ ] `temporal_refinement` _(required)_: `static`
- [ ] `x_resolution`: `1.875`
- [ ] `y_resolution`: `1.25`
- [ ] `horizontal_units`
- [ ] `resolution_range_km`
- [ ] `spatial_refinement`
- [ ] `units`: `degree` ← extra


> [!CAUTION]
> **Schema validation failed** — the following errors must be resolved before this submission can be merged.
>
> | **Field** | **Error Type** | **Input Value** | **Input Type** | **Message** |
> | --- | --- | --- | --- | --- |
> | region.Region | model_type | `None` | None | Input should be a valid dictionary or instance of Region |
> | region.str | string_type | `None` | None | Input should be a valid string |
> | horizontal_units | value_error | `None` | None | Value error, horizontal_units is required when x_resolution or y_resolution are set |


---

### 2. Controlled Vocabulary Links

```
We are able to compare the controlled aspect of a submission by looking at the links to registered components of the CVs as provided by the dropdown fields. This is the quickest way to identify potential duplicates and overlaps between submissions.
```

**Checking that linked files resolve: 5/5 (100%)**

<details><summary>Graph of links in submission.</summary>

```mermaid
graph TD
    tempgrid_matthew_mizielinski_1776857536(["tempgrid-matthew-mizielinski-1776857536"])

    subgraph sg_grid_mapping["grid_mapping"]
        grid_mapping_latitude_longitude["latitude-longitude"]
        click grid_mapping_latitude_longitude "https://constants.mipcvs.dev/grid_mapping/latitude-longitude.json" _blank
    end

    tempgrid_matthew_mizielinski_1776857536 --> grid_mapping_latitude_longitude
    subgraph sg_grid_type["grid_type"]
        grid_type_regular_latitude_longitude["regular-latitude-longitude"]
        click grid_type_regular_latitude_longitude "https://constants.mipcvs.dev/grid_type/regular-latitude-longitude.json" _blank
    end

    tempgrid_matthew_mizielinski_1776857536 --> grid_type_regular_latitude_longitude
    subgraph sg_region["region"]
        region_global["global"]
        click region_global "https://constants.mipcvs.dev/region/global.json" _blank
    end

    tempgrid_matthew_mizielinski_1776857536 --> region_global
    subgraph sg_temporal_refinement["temporal_refinement"]
        temporal_refinement_static["static"]
        click temporal_refinement_static "https://constants.mipcvs.dev/temporal_refinement/static.json" _blank
    end

    tempgrid_matthew_mizielinski_1776857536 --> temporal_refinement_static
    subgraph sg_units["units"]
        units_degree["degree"]
        click units_degree "https://constants.mipcvs.dev/units/degree.json" _blank
    end

    tempgrid_matthew_mizielinski_1776857536 --> units_degree
```

</details>

> [!WARNING]
> **11 existing item(s) share ≥75% link overlap.** Review field differences below before merging.

- [ ] [`g115`](https://emd.mipcvs.dev/horizontal_grid_cell/g115.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 77.0%

<div style="padding-left:1.5em"><details><summary>Compare against g115</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `units` | _null_ | degree |

</details></div>

- [ ] [`g108`](https://emd.mipcvs.dev/horizontal_grid_cell/g108.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 87.8%

<div style="padding-left:1.5em"><details><summary>Compare against g108</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `units` | _null_ | degree |

</details></div>

- [ ] [`g106`](https://emd.mipcvs.dev/horizontal_grid_cell/g106.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 71.0%

<div style="padding-left:1.5em"><details><summary>Compare against g106</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 64800 | 27648 |
| `southernmost_latitude` | -89.5 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | 1 | 1.875 |
| `y_resolution` | 1 | 1.25 |

</details></div>

- [ ] [`g117`](https://emd.mipcvs.dev/horizontal_grid_cell/g117.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 79.2%

<div style="padding-left:1.5em"><details><summary>Compare against g117</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | hierarchical-discrete-global-grid | regular-latitude-longitude |
| `n_cells` | _null_ | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9424 | 1.25 |

</details></div>

- [ ] [`g100`](https://emd.mipcvs.dev/horizontal_grid_cell/g100.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 70.0%

<div style="padding-left:1.5em"><details><summary>Compare against g100</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 55296 | 27648 |
| `southernmost_latitude` | -89.5 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9 | 1.25 |

</details></div>

- [ ] [`g110`](https://emd.mipcvs.dev/horizontal_grid_cell/g110.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 88.5%

<div style="padding-left:1.5em"><details><summary>Compare against g110</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g113`](https://emd.mipcvs.dev/horizontal_grid_cell/g113.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 75.8%

<div style="padding-left:1.5em"><details><summary>Compare against g113</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 1036800 | 27648 |
| `southernmost_latitude` | -89.875 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.125 | 0 |
| `x_resolution` | 0.25 | 1.875 |
| `y_resolution` | 0.25 | 1.25 |

</details></div>

- [ ] [`g109`](https://emd.mipcvs.dev/horizontal_grid_cell/g109.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 78.6%

<div style="padding-left:1.5em"><details><summary>Compare against g109</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g114`](https://emd.mipcvs.dev/horizontal_grid_cell/g114.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 83.4%

<div style="padding-left:1.5em"><details><summary>Compare against g114</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | regular-gaussian | regular-latitude-longitude |
| `n_cells` | 131072 | 27648 |
| `southernmost_latitude` | -89.4628215685774 | -89.375 |
| `x_resolution` | 0.7 | 1.875 |
| `y_resolution` | 0.7 | 1.25 |

</details></div>

- [ ] [`g105`](https://emd.mipcvs.dev/horizontal_grid_cell/g105.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 69.0%

<div style="padding-left:1.5em"><details><summary>Compare against g105</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 32768 | 27648 |
| `southernmost_latitude` | -88.9277353522076 | -89.375 |
| `units` | _null_ | degree |
| `x_resolution` | 1.4 | 1.875 |
| `y_resolution` | 1.4 | 1.25 |

</details></div>

- [ ] [`g111`](https://emd.mipcvs.dev/horizontal_grid_cell/g111.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content similarity: 67.8%

<div style="padding-left:1.5em"><details><summary>Compare against g111</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 20592 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `units` | _null_ | degree |
| `x_resolution` | 2.5 | 1.875 |
| `y_resolution` | 1.3 | 1.25 |

</details></div>


<details><summary>All CV link comparisons</summary>

- [ ] [`g115`](https://emd.mipcvs.dev/horizontal_grid_cell/g115.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 77.0%

<div style="padding-left:1.5em"><details><summary>Compare against g115</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `units` | _null_ | degree |

</details></div>

- [ ] [`g108`](https://emd.mipcvs.dev/horizontal_grid_cell/g108.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 87.8%

<div style="padding-left:1.5em"><details><summary>Compare against g108</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `units` | _null_ | degree |

</details></div>

- [ ] [`g106`](https://emd.mipcvs.dev/horizontal_grid_cell/g106.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 71.0%

<div style="padding-left:1.5em"><details><summary>Compare against g106</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 64800 | 27648 |
| `southernmost_latitude` | -89.5 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | 1 | 1.875 |
| `y_resolution` | 1 | 1.25 |

</details></div>

- [ ] [`g117`](https://emd.mipcvs.dev/horizontal_grid_cell/g117.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 79.2%

<div style="padding-left:1.5em"><details><summary>Compare against g117</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | hierarchical-discrete-global-grid | regular-latitude-longitude |
| `n_cells` | _null_ | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9424 | 1.25 |

</details></div>

- [ ] [`g100`](https://emd.mipcvs.dev/horizontal_grid_cell/g100.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 70.0%

<div style="padding-left:1.5em"><details><summary>Compare against g100</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 55296 | 27648 |
| `southernmost_latitude` | -89.5 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9 | 1.25 |

</details></div>

- [ ] [`g110`](https://emd.mipcvs.dev/horizontal_grid_cell/g110.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 88.5%

<div style="padding-left:1.5em"><details><summary>Compare against g110</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g113`](https://emd.mipcvs.dev/horizontal_grid_cell/g113.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 75.8%

<div style="padding-left:1.5em"><details><summary>Compare against g113</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 1036800 | 27648 |
| `southernmost_latitude` | -89.875 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.125 | 0 |
| `x_resolution` | 0.25 | 1.875 |
| `y_resolution` | 0.25 | 1.25 |

</details></div>

- [ ] [`g109`](https://emd.mipcvs.dev/horizontal_grid_cell/g109.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 78.6%

<div style="padding-left:1.5em"><details><summary>Compare against g109</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g114`](https://emd.mipcvs.dev/horizontal_grid_cell/g114.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 83.4%

<div style="padding-left:1.5em"><details><summary>Compare against g114</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | regular-gaussian | regular-latitude-longitude |
| `n_cells` | 131072 | 27648 |
| `southernmost_latitude` | -89.4628215685774 | -89.375 |
| `x_resolution` | 0.7 | 1.875 |
| `y_resolution` | 0.7 | 1.25 |

</details></div>

- [ ] [`g105`](https://emd.mipcvs.dev/horizontal_grid_cell/g105.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 69.0%

<div style="padding-left:1.5em"><details><summary>Compare against g105</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 32768 | 27648 |
| `southernmost_latitude` | -88.9277353522076 | -89.375 |
| `units` | _null_ | degree |
| `x_resolution` | 1.4 | 1.875 |
| `y_resolution` | 1.4 | 1.25 |

</details></div>

- [ ] [`g111`](https://emd.mipcvs.dev/horizontal_grid_cell/g111.json) — Links: 4/5 (80.0%) `████████░░`  ·  Content: 67.8%

<div style="padding-left:1.5em"><details><summary>Compare against g111</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 20592 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `units` | _null_ | degree |
| `x_resolution` | 2.5 | 1.875 |
| `y_resolution` | 1.3 | 1.25 |

</details></div>

- [ ] [`g118`](https://emd.mipcvs.dev/horizontal_grid_cell/g118.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 78.7%

<div style="padding-left:1.5em"><details><summary>Compare against g118</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `description` | South of 65°N, the resolution of the model is 1° along the z… | _null_ |
| `grid_type` | tripolar | regular-latitude-longitude |
| `n_cells` | 108000 | 27648 |
| `southernmost_latitude` | -77.8766 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g112`](https://emd.mipcvs.dev/horizontal_grid_cell/g112.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 74.8%

<div style="padding-left:1.5em"><details><summary>Compare against g112</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | tripolar | regular-latitude-longitude |
| `horizontal_units` | degree | _null_ |
| `n_cells` | 120184 | 27648 |
| `southernmost_latitude` | -85.789 | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | 180 | 0 |
| `x_resolution` | 1 | 1.875 |
| `y_resolution` | 1 | 1.25 |

</details></div>

- [ ] [`g103`](https://emd.mipcvs.dev/horizontal_grid_cell/g103.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 71.6%

<div style="padding-left:1.5em"><details><summary>Compare against g103</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | tripolar | regular-latitude-longitude |
| `horizontal_units` | _null_ | _null_ |
| `n_cells` | 105705 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g102`](https://emd.mipcvs.dev/horizontal_grid_cell/g102.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 68.5%

<div style="padding-left:1.5em"><details><summary>Compare against g102</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `description` | Wrap-around | _null_ |
| `grid_type` | tripolar | regular-latitude-longitude |
| `horizontal_units` | _null_ | _null_ |
| `n_cells` | 105704 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g107`](https://emd.mipcvs.dev/horizontal_grid_cell/g107.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 68.7%

<div style="padding-left:1.5em"><details><summary>Compare against g107</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | tripolar | regular-latitude-longitude |
| `horizontal_units` | _null_ | _null_ |
| `n_cells` | 120184 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g116`](https://emd.mipcvs.dev/horizontal_grid_cell/g116.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 77.9%

<div style="padding-left:1.5em"><details><summary>Compare against g116</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `description` | South of 65°N, the resolution of the model is 1° along the z… | _null_ |
| `grid_type` | tripolar | regular-latitude-longitude |
| `n_cells` | 108000 | 27648 |
| `southernmost_latitude` | -77.7532 | -89.375 |
| `units` | _null_ | degree |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g101`](https://emd.mipcvs.dev/horizontal_grid_cell/g101.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 72.9%

<div style="padding-left:1.5em"><details><summary>Compare against g101</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | linear-spectral-gaussian | regular-latitude-longitude |
| `horizontal_units` | degree | _null_ |
| `n_cells` | 24572 | 27648 |
| `southernmost_latitude` | -88.9277353522076 | -89.375 |
| `truncation_method` | triangular | _null_ |
| `truncation_number` | 127 | _null_ |
| `units` | _null_ | degree |
| `x_resolution` | 1.4 | 1.875 |
| `y_resolution` | 1.4 | 1.25 |

</details></div>

- [ ] [`g104`](https://emd.mipcvs.dev/horizontal_grid_cell/g104.json) — Links: 3/5 (60.0%) `██████░░░░`  ·  Content: 69.8%

<div style="padding-left:1.5em"><details><summary>Compare against g104</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `grid_type` | tripolar | regular-latitude-longitude |
| `horizontal_units` | _null_ | _null_ |
| `n_cells` | 105706 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `units` | _null_ | degree |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>


</details>


---

### 3. Content Similarity

```
When it comes to free text and numerical entries, direct comparisons are more difficult. Instead we use a combination of text similarity and field exclusions to identify items that contain similar content. We exclude fields that carry links (they are covered by the previous section) and fields that have explicit pydantic validators (they have explicit checks) to focus on content that is not already being checked by other means.
```

_7 field(s) compared using embedding._

<details><summary>Fields included in comparison</summary>

| Field | Value |
|-------|-------|
| `n_cells` | 27648 |
| `southernmost_latitude` | -89.375 |
| `truncation_method` |  |
| `truncation_number` |  |
| `westernmost_longitude` | 0 |
| `x_resolution` | 1.875 |
| `y_resolution` | 1.25 |

</details>

> [!WARNING]
> **9 existing item(s) exceed 75% content similarity.** Confirm this submission is not a duplicate.

- [ ] [`g110`](https://emd.mipcvs.dev/horizontal_grid_cell/g110.json) — 88.5% `████████░░`

<div style="padding-left:1.5em"><details><summary>Compare against g110</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g108`](https://emd.mipcvs.dev/horizontal_grid_cell/g108.json) — 87.8% `████████░░`

<div style="padding-left:1.5em"><details><summary>Compare against g108</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |

</details></div>

- [ ] [`g114`](https://emd.mipcvs.dev/horizontal_grid_cell/g114.json) — 83.4% `████████░░`

<div style="padding-left:1.5em"><details><summary>Compare against g114</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 131072 | 27648 |
| `southernmost_latitude` | -89.4628215685774 | -89.375 |
| `x_resolution` | 0.7 | 1.875 |
| `y_resolution` | 0.7 | 1.25 |

</details></div>

- [ ] [`g117`](https://emd.mipcvs.dev/horizontal_grid_cell/g117.json) — 79.2% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g117</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | _null_ | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9424 | 1.25 |

</details></div>

- [ ] [`g118`](https://emd.mipcvs.dev/horizontal_grid_cell/g118.json) — 78.7% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g118</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 108000 | 27648 |
| `southernmost_latitude` | -77.8766 | -89.375 |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g109`](https://emd.mipcvs.dev/horizontal_grid_cell/g109.json) — 78.6% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g109</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g116`](https://emd.mipcvs.dev/horizontal_grid_cell/g116.json) — 77.9% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g116</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 108000 | 27648 |
| `southernmost_latitude` | -77.7532 | -89.375 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g115`](https://emd.mipcvs.dev/horizontal_grid_cell/g115.json) — 77.0% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g115</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |

</details></div>

- [ ] [`g113`](https://emd.mipcvs.dev/horizontal_grid_cell/g113.json) — 75.8% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g113</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 1036800 | 27648 |
| `southernmost_latitude` | -89.875 | -89.375 |
| `westernmost_longitude` | 0.125 | 0 |
| `x_resolution` | 0.25 | 1.875 |
| `y_resolution` | 0.25 | 1.25 |

</details></div>


<details><summary>All content comparisons</summary>

- [ ] [`g110`](https://emd.mipcvs.dev/horizontal_grid_cell/g110.json) — 88.5% `████████░░`

<div style="padding-left:1.5em"><details><summary>Compare against g110</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g108`](https://emd.mipcvs.dev/horizontal_grid_cell/g108.json) — 87.8% `████████░░`

<div style="padding-left:1.5em"><details><summary>Compare against g108</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |

</details></div>

- [ ] [`g114`](https://emd.mipcvs.dev/horizontal_grid_cell/g114.json) — 83.4% `████████░░`

<div style="padding-left:1.5em"><details><summary>Compare against g114</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 131072 | 27648 |
| `southernmost_latitude` | -89.4628215685774 | -89.375 |
| `x_resolution` | 0.7 | 1.875 |
| `y_resolution` | 0.7 | 1.25 |

</details></div>

- [ ] [`g117`](https://emd.mipcvs.dev/horizontal_grid_cell/g117.json) — 79.2% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g117</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | _null_ | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9424 | 1.25 |

</details></div>

- [ ] [`g118`](https://emd.mipcvs.dev/horizontal_grid_cell/g118.json) — 78.7% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g118</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 108000 | 27648 |
| `southernmost_latitude` | -77.8766 | -89.375 |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g109`](https://emd.mipcvs.dev/horizontal_grid_cell/g109.json) — 78.6% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g109</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `westernmost_longitude` | 0.9375 | 0 |

</details></div>

- [ ] [`g116`](https://emd.mipcvs.dev/horizontal_grid_cell/g116.json) — 77.9% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g116</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 108000 | 27648 |
| `southernmost_latitude` | -77.7532 | -89.375 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g115`](https://emd.mipcvs.dev/horizontal_grid_cell/g115.json) — 77.0% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g115</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 27840 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |

</details></div>

- [ ] [`g113`](https://emd.mipcvs.dev/horizontal_grid_cell/g113.json) — 75.8% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g113</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 1036800 | 27648 |
| `southernmost_latitude` | -89.875 | -89.375 |
| `westernmost_longitude` | 0.125 | 0 |
| `x_resolution` | 0.25 | 1.875 |
| `y_resolution` | 0.25 | 1.25 |

</details></div>

- [ ] [`g112`](https://emd.mipcvs.dev/horizontal_grid_cell/g112.json) — 74.8% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g112</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 120184 | 27648 |
| `southernmost_latitude` | -85.789 | -89.375 |
| `westernmost_longitude` | 180 | 0 |
| `x_resolution` | 1 | 1.875 |
| `y_resolution` | 1 | 1.25 |

</details></div>

- [ ] [`g101`](https://emd.mipcvs.dev/horizontal_grid_cell/g101.json) — 72.9% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g101</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 24572 | 27648 |
| `southernmost_latitude` | -88.9277353522076 | -89.375 |
| `truncation_number` | 127 | _null_ |
| `x_resolution` | 1.4 | 1.875 |
| `y_resolution` | 1.4 | 1.25 |

</details></div>

- [ ] [`g103`](https://emd.mipcvs.dev/horizontal_grid_cell/g103.json) — 71.6% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g103</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 105705 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g106`](https://emd.mipcvs.dev/horizontal_grid_cell/g106.json) — 71.0% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g106</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 64800 | 27648 |
| `southernmost_latitude` | -89.5 | -89.375 |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | 1 | 1.875 |
| `y_resolution` | 1 | 1.25 |

</details></div>

- [ ] [`g100`](https://emd.mipcvs.dev/horizontal_grid_cell/g100.json) — 70.0% `███████░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g100</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 55296 | 27648 |
| `southernmost_latitude` | -89.5 | -89.375 |
| `westernmost_longitude` | 0.5 | 0 |
| `x_resolution` | 1.25 | 1.875 |
| `y_resolution` | 0.9 | 1.25 |

</details></div>

- [ ] [`g104`](https://emd.mipcvs.dev/horizontal_grid_cell/g104.json) — 69.8% `██████░░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g104</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 105706 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g105`](https://emd.mipcvs.dev/horizontal_grid_cell/g105.json) — 69.0% `██████░░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g105</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 32768 | 27648 |
| `southernmost_latitude` | -88.9277353522076 | -89.375 |
| `x_resolution` | 1.4 | 1.875 |
| `y_resolution` | 1.4 | 1.25 |

</details></div>

- [ ] [`g107`](https://emd.mipcvs.dev/horizontal_grid_cell/g107.json) — 68.7% `██████░░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g107</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 120184 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g102`](https://emd.mipcvs.dev/horizontal_grid_cell/g102.json) — 68.5% `██████░░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g102</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `n_cells` | 105704 | 27648 |
| `southernmost_latitude` | _null_ | -89.375 |
| `westernmost_longitude` | _null_ | 0 |
| `x_resolution` | _null_ | 1.875 |
| `y_resolution` | _null_ | 1.25 |

</details></div>

- [ ] [`g111`](https://emd.mipcvs.dev/horizontal_grid_cell/g111.json) — 67.8% `██████░░░░`

<div style="padding-left:1.5em"><details><summary>Compare against g111</summary>

| Field | Existing | Submitted |
|-------|----------|-----------|
| `horizontal_units` | degree | _null_ |
| `n_cells` | 20592 | 27648 |
| `southernmost_latitude` | -90 | -89.375 |
| `x_resolution` | 2.5 | 1.875 |
| `y_resolution` | 1.3 | 1.25 |

</details></div>


</details>


---
_Generated by [cmipld](https://github.com/WCRP-CMIP/CMIP-LD) · 2026-04-28 00:35 UTC_


## 7. Reload & Rerun After Code Changes
Use this cell to quickly re-test after editing report_builder.py.

In [ ]:
importlib.reload(rb_mod)
from cmipld.utils.similarity.report_builder import (
    ReportBuilder, _build_links_from_folder,
    _infer_cv_graphs_from_folder, _fetch_cv_graph,
)

field_links, field_graphs = _build_links_from_folder(TEST_ITEM, folder_items)
print(f'Links: {len(field_links)}/{len(field_graphs)}')
for k, v in field_links.items():
    print(f'  {k}: {v.split("/")[-1]}')

[Cache HIT] https://constants.mipcvs.dev/grid_mapping/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/grid_type/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/region/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/temporal_refinement/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/units/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/truncation_method/_graph.json (depth=2)
Links: 5/6
  grid_mapping: latitude-longitude
  grid_type: regular-latitude-longitude
  region: global
  temporal_refinement: static
  units: degree


In [ ]:
# cmipld.get('emd:horizontal_grid_cell/_graph.json')
# cmipld.client.cache_clear()

Cache cleared: 16 entries removed


16

In [ ]:
# !mamba install pandas --yes


In [ ]:
# import yaml
# import glob
# from openpyxl import Workbook
# from openpyxl.worksheet.datavalidation import DataValidation

# yaml_files = glob.glob('/Users/daniel.ellis/WIPwork/Essential-Model-Documentation/.github/ISSUE_TEMPLATE/*.yml')

# for yaml_file in yaml_files:
#     print(f"Processing {yaml_file}...")
    
#     if 'config' in yaml_file:
#         print("  Skipping config file.")
#         continue

#     with open(yaml_file, "r", encoding="utf-8") as f:
#         form = yaml.safe_load(f)

#     wb = Workbook()
#     ws = wb.active
#     ws.title = "Form"

#     fields = []
#     dropdowns = {}

#     # Extract fields
#     for item in form.get("body", []):
#         if "id" in item:
#             field_id = item["id"]
#             label = item.get("attributes", {}).get("label", field_id)
#             options = item.get("attributes", {}).get("options", [])

#             fields.append(label)

#             if options:
#                 dropdowns[label] = options

#     # Write header row
#     for col, field in enumerate(fields, start=1):
#         ws.cell(row=1, column=col, value=field)

#     # Add dropdowns (apply to first 100 rows for usability)
#     for col, field in enumerate(fields, start=1):
#         if field in dropdowns:
#             options = dropdowns[field]

#             dv = DataValidation(
#                 type="list",
#                 formula1=f'"{",".join(options)}"',
#                 allow_blank=True
#             )

#             ws.add_data_validation(dv)

#             # Apply dropdown to rows 2–100
#             col_letter = ws.cell(row=1, column=col).column_letter
#             dv.add(f"{col_letter}2:{col_letter}100")

#     # Save file
#     output_file = yaml_file.split('/')[-1].replace(".yml", ".xlsx")
#     wb.save(output_file)

#     print(f"  Created {output_file}")